# Lesson 3.1 — Auto Scaling Groups & Elastic Load Balancing

**Level: Intermediate** | **Exam:** SOA-C03 Domain 2, Task 2.1

## Learning Objectives
- Configure Auto Scaling Groups (ASG) with launch templates
- Create Application Load Balancers with target groups and health checks
- Implement target tracking, step, and scheduled scaling policies
- Select ALB vs NLB vs GLB for specific use cases

_Reference: [ELB Docs](https://docs.aws.amazon.com/elasticloadbalancing/latest/userguide/) | [ASG Docs](https://docs.aws.amazon.com/autoscaling/ec2/userguide/)_

## Load Balancer Types

| Type | Layer | Use Case |
|------|-------|----------|
| **ALB** | 7 (HTTP) | Web apps, microservices, gRPC, WebSocket |
| **NLB** | 4 (TCP/UDP) | Gaming, IoT, ultra-low latency, static IPs |
| **GLB** | 3+4 | Network appliances (inline firewalls, IDS) |

## Auto Scaling Policies

| Policy | How it works | When to use |
|--------|-------------|-------------|
| **Target Tracking** | Keeps metric at target (e.g., CPU=50%) | Recommended for most workloads |
| **Step Scaling** | Scale by different amounts based on alarm breach size | When you need precise control |
| **Scheduled** | Pre-scale at specific times | Known traffic patterns (rush hour) |
| **Predictive** | ML-based proactive scaling | Variable but recurring traffic |

## Health Check Types
- **EC2 health check** (default): instance stops → replace it
- **ELB health check**: target group health check fails → replace it  
- **Always use ELB health checks** for apps behind an ALB/NLB

In [ ]:
import boto3, json

autoscaling = boto3.client('autoscaling', region_name='us-east-1')
elbv2 = boto3.client('elbv2', region_name='us-east-1')
ec2 = boto3.client('ec2', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# Visualize scaling policy structures

print('=== Target Tracking Scaling Policy ===')
target_tracking = {
    'PolicyType': 'TargetTrackingScaling',
    'TargetTrackingScalingPolicyConfiguration': {
        'TargetValue': 50.0,
        'PredefinedMetricSpecification': {
            'PredefinedMetricType': 'ASGAverageCPUUtilization'
        },
        'ScaleInCooldown': 300,
        'ScaleOutCooldown': 60
    }
}
print(json.dumps(target_tracking, indent=2))

print('\n=== Other Predefined Metrics for Target Tracking ===')
metrics = [
    'ASGAverageCPUUtilization',
    'ASGAverageNetworkIn',
    'ASGAverageNetworkOut',
    'ALBRequestCountPerTarget'   # Requires ALB ARN in customization
]
for m in metrics:
    print(f'  {m}')

In [ ]:
# List existing load balancers
response = elbv2.describe_load_balancers()
lbs = response['LoadBalancers']

if lbs:
    print(f'Load Balancers ({len(lbs)}):')
    for lb in lbs:
        print(f'  {lb["LoadBalancerName"]:35} {lb["Type"]:10} {lb["State"]["Code"]}')
        tgs = elbv2.describe_target_groups(LoadBalancerArn=lb['LoadBalancerArn'])['TargetGroups']
        for tg in tgs:
            print(f'    → Target Group: {tg["TargetGroupName"]} ({tg["Protocol"]}:{tg["Port"]})')
else:
    print('No load balancers found.')
    print('This is expected in a fresh account. Launch resources in later lessons.')

In [ ]:
# List existing Auto Scaling Groups and their policies
asgs = autoscaling.describe_auto_scaling_groups()['AutoScalingGroups']

if asgs:
    for asg in asgs:
        print(f'ASG: {asg["AutoScalingGroupName"]}')
        print(f'  Desired: {asg["DesiredCapacity"]} | Min: {asg["MinSize"]} | Max: {asg["MaxSize"]}')
        print(f'  AZs: {asg["AvailabilityZones"]}')
        print(f'  Health Check: {asg["HealthCheckType"]}')
else:
    print('No ASGs found.')

## Exercise

In [ ]:
# TODO: For each scenario below, select the appropriate load balancer type and scaling policy
# Fill in your answers in the dict

scenarios = [
    {
        'scenario': 'E-commerce site with predictable Black Friday peak (10x normal)',
        'lb_type': 'TODO',       # ALB / NLB / GLB
        'scaling': 'TODO',       # Target Tracking / Step / Scheduled / Predictive
        'reason': 'TODO'
    },
    {
        'scenario': 'Real-time multiplayer game using UDP, needs static IP',
        'lb_type': 'TODO',
        'scaling': 'TODO',
        'reason': 'TODO'
    },
    {
        'scenario': 'Corporate web app with unpredictable CPU spikes',
        'lb_type': 'TODO',
        'scaling': 'TODO',
        'reason': 'TODO'
    },
]

for s in scenarios:
    print(f"Scenario: {s['scenario']}")
    print(f"  LB Type : {s['lb_type']}")
    print(f"  Scaling : {s['scaling']}")
    print(f"  Reason  : {s['reason']}")
    print()

## Summary
- **ALB** for HTTP/HTTPS apps; **NLB** for TCP/UDP ultra-performance; **GLB** for inline appliances
- **Target Tracking** is the recommended default scaling policy — AWS handles the math
- Use **ELB health checks** (not EC2) when instances serve traffic behind an ALB/NLB
- Cooldown periods prevent scaling oscillation — scale-out cooldown should be shorter than scale-in

---
**Next:** [Lesson 3.2 — Multi-AZ & Multi-Region Architectures](lesson-3.2-multi-az-region.ipynb)